# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebulhaq02/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muneebulhaq02/flyrank-ml-internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())


Working dir: /content/FlyRank-Internship/FlyRank-Internship


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline ranks pages for review using simple transparent rules instead of machine learning.

A page receives a higher priority when it:

- has many Google Search impressions,
- is older content,
- has a lower average search ranking (higher numeric position),
- receives relatively fewer clicks compared to impressions.

These conditions are combined into one baseline score.

### Reason Codes

Each recommendation includes one or more reason codes.

- High Visibility → Many impressions
- Old Content → Content age exceeds threshold
- Low CTR → CTR below average
- Poor Position → Average search position is relatively low

These reason codes explain why a page appears in the ranked review list.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

baseline_df = df.copy()

# Binary rule conditions

baseline_df["high_visibility"] = (
    baseline_df["impressions_90d"] >= baseline_df["impressions_90d"].median()
).astype(int)

baseline_df["old_content"] = (
    baseline_df["content_age_days"] >= baseline_df["content_age_days"].median()
).astype(int)

baseline_df["poor_position"] = (
    baseline_df["avg_position"] >= baseline_df["avg_position"].median()
).astype(int)

baseline_df["low_ctr"] = (
    baseline_df["ctr"] <= baseline_df["ctr"].median()
).astype(int)

baseline_df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,high_visibility,old_content,poor_position,low_ctr
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,4.55,0.0,good,striking,down,-41.4,1,0,0,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,10.00,0.0,good,page_3_5,down,-57.7,1,1,1,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,28.57,0.0,good,page_3_5,down,-60.9,1,0,1,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,3.45,0.0,good,page_1,stable,-13.8,1,1,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,24.29,0.0,good,page_3_5,down,-34.7,1,1,1,0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Building the Baseline Score

Each rule contributes one point.

Pages with more matching conditions receive higher scores.

The ranked output is saved for later comparison with the machine learning model.

This baseline acts as a transparent decision-support system.

In [7]:
baseline_df["baseline_score"] = (
      baseline_df["high_visibility"]
    + baseline_df["old_content"]
    + baseline_df["poor_position"]
    + baseline_df["low_ctr"]
)

baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
)

import os

os.makedirs("work/outputs", exist_ok=True)

baseline_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(baseline_df))
print()

baseline_df[
    [
        "baseline_score",
        "impressions_90d",
        "content_age_days",
        "avg_position",
        "ctr"
    ]
].head(10)

Rows: 30000



,baseline_score,impressions_90d,content_age_days,avg_position,ctr
8819,4,763,445,29.2,0.00
8817,4,812,256,23.5,0.00
8841,4,801,482,55.5,0.00
8832,4,7546,300,39.4,0.03
26182,4,1543,419,12.8,0.00
26181,4,938,271,80.0,0.00
21032,4,12109,419,13.4,0.04
21086,4,19242,421,11.4,0.03
62,4,38542,299,33.4,0.05
21054,4,19976,286,23.9,0.05


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The twenty highest-ranked pages were manually reviewed.

Each recommendation contains:

- suggested action,
- reason code,
- confidence,
- possible limitations.

This review checks whether the baseline behaves reasonably before introducing machine learning.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline_df.head(20).copy()

top20["Suggested Action"] = "Review content"

top20["Reason Code"] = ""

top20.loc[top20["high_visibility"] == 1, "Reason Code"] += "High Visibility; "
top20.loc[top20["old_content"] == 1, "Reason Code"] += "Old Content; "
top20.loc[top20["poor_position"] == 1, "Reason Code"] += "Poor Position; "
top20.loc[top20["low_ctr"] == 1, "Reason Code"] += "Low CTR; "

top20["Confidence"] = np.where(
    top20["baseline_score"] >= 3,
    "High",
    "Medium"
)

top20["What Could Make It Wrong"] = (
    "Ranking changes caused by factors not included in this dataset."
)

top20[
    [
        "baseline_score",
        "Suggested Action",
        "Reason Code",
        "Confidence",
        "What Could Make It Wrong"
    ]
]

,baseline_score,Suggested Action,Reason Code,Confidence,What Could Make It Wrong
8819,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
8817,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
8841,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
8832,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
26182,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
26181,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
21032,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
21086,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
62,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...
21054,4,Review content,High Visibility; Old Content; Poor Position; L...,High,Ranking changes caused by factors not included...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

Some pages may receive a high baseline score because they satisfy several simple rules even though they are not true refresh opportunities.

Examples include pages with naturally low CTR or seasonal changes.

No product flags, future information, label-derived columns, or client identifiers were used when computing the baseline score.

Therefore, the baseline remains interpretable and free from known leakage sources.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Top score distribution")
print(baseline_df["baseline_score"].value_counts().sort_index())

print()

excluded_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "client_id",
    "content_id"
]

print("Leakage-sensitive columns intentionally excluded:")

for col in excluded_columns:
    print("-", col)

print()

print("Baseline CSV saved successfully:")
print("work/outputs/baseline_action_score.csv")

Top score distribution
baseline_score
0      933
1     7990
2    12288
3     7263
4     1526
Name: count, dtype: int64

Leakage-sensitive columns intentionally excluded:
- trend_pct
- trend_direction
- is_declining_label
- client_id
- content_id

Baseline CSV saved successfully:
work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.